# CPMK 3 — Pelatihan dan evaluasi tiga kelas tersulit

Eksperimen ini membandingkan YOLO11n-seg, YOLOv8n-seg, dan Mask R-CNN ResNet-50 FPN pada **bed**, **book**, dan **cabinet-merged**. Ketiga model dimulai dari bobot pralatih baseline. Perubahan utama adalah kelas yang diberi anotasi pelatihan. Seluruh gambar dipertahankan, termasuk gambar tanpa objek target.

Gunakan **GPU Tesla T4, satu GPU**, dan aktifkan **Internet** untuk memasang dependensi dan memperoleh bobot pralatih. Tambahkan **Panoptic Indoor Segmentation v5 – COCO format** versi baseline, serta satu Kaggle Dataset berisi `evaluation_config.json` dan `per_class_ap_comparison.csv` dari `analysis/all_class_comparison/`. Keduanya juga dapat diberikan melalui ZIP evaluasi asli. Bobot all-class tidak diperlukan. Impor notebook, periksa pengaturan, lalu pilih **Save Version → Save & Run All**.

H1 membandingkan perubahan mean AP tiga kelas dalam setiap arsitektur. H2 membandingkan mean AP antar-model Hard-3. H3 membandingkan AP *book* antar-model Hard-3. Tabel notebook hanya memakai validation. Pengujian akhir pada original test dan personal test dilakukan terpisah.

Hasil ditulis ke folder run yang baru di `/kaggle/working/cpmk3_hard3/`. Untuk memulihkan model yang selesai, pasang ZIP individual dari output versi Kaggle atau dari unduhan sebagai input dan isi `RECOVERY_ZIPS`. Artefak yang belum disimpan sebagai output versi atau diunduh bisa hilang saat sesi berakhir. Model yang gagal dapat dijalankan ulang dari bobot awal dengan mengaktifkan flag model tersebut saja. ZIP pemulihan memakai layout notebook ini.

## 1. Dependensi dan lingkungan

PyTorch dan Torchvision memakai pasangan yang disediakan Kaggle. Ultralytics dipatok ke versi baseline. Versi aktual dan GPU dicatat agar perbedaan lingkungan dapat ditelusuri.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics==8.4.216", "pycocotools==2.0.10", "pyyaml"],
    check=True,
)

import csv
import gc
import hashlib
import io
import json
import math
import random
import shutil
import time
import traceback
from collections import Counter, defaultdict
from contextlib import redirect_stdout
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZIP_STORED, ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import ultralytics
import yaml
from IPython.display import display
from PIL import Image
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights, maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan satu GPU Tesla T4 di pengaturan Kaggle.")
DEVICE = torch.device("cuda:0")
assert ultralytics.__version__ == "8.4.216", "Mulai ulang kernel setelah mengganti versi Ultralytics."
torchvision.ops.nms(torch.tensor([[0., 0., 3., 3.]], device=DEVICE),
                    torch.ones(1, device=DEVICE), 0.5)

## 2. Konfigurasi dan pilihan eksekusi

YOLO mempertahankan AdamW yang dipilih saat baseline (learning rate 0,000128 dan momentum 0,9). Nilai tersebut dibuat eksplisit karena `optimizer=auto` menyesuaikan learning rate terhadap jumlah kelas. Mask R-CNN memakai SGD dan MultiStepLR seperti baseline. Evaluasi final validation memuat ulang checkpoint terbaik, dengan kriteria bawaan Ultralytics untuk YOLO dan mask mAP50–95 validation untuk Mask R-CNN.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
DATASET_ROOT = None                 # Isi jika ada beberapa salinan dataset.
BASELINE_SOURCE = None              # Folder dua berkas evaluasi atau ZIP evaluasi.
RUN_TRAIN = {"yolo11n": True, "yolov8n": True, "maskrcnn": True}
RUN_EVALUATION = True
RECOVERY_ZIPS = {}                   # Contoh: {"yolo11n": "/kaggle/input/.../yolo11n.zip"}
PRETRAINED_ROOT = None              # Folder opsional dengan bobot YOLO asli.
SEED = 42
CLASSES = {2: (1, "bed"), 4: (2, "book"), 8: (3, "cabinet-merged")}
NAMES = [name for _, name in CLASSES.values()]
EXPECTED = {"train": (397, 178, 60), "valid": (166, 40, 19), "test": (359, 530, 137)}
SOURCE_HASHES = {
    "train": "6426839e66216ad207e3f51f9521520256ec5aa33effe57a06a5d3c766b49117",
    "valid": "e0ca4ae5d12ce52a16445e9395a244424520c0b8c0c635201deb68257e474274",
    "test": "1b756f0cab2be19fb338e35aea80c3a1e40eca6c5343f4daee9bf3d675bd466f",
}
ROTATIONS = {
    ("train", "182169107_jpg.rf.c9b1bbecf1474a5b36a6df197c30dfed.jpg"): "CCW",
    ("train", "182169121_jpg.rf.eb0ccb71d0ea38adf1e38bf9c02a4a15.jpg"): "CCW",
    ("valid", "182169109_jpg.rf.0531e60eb6e404d9633ed0c23a381fe7.jpg"): "CW",
    ("valid", "182169108_jpg.rf.205d9f1ed8f888006b8f97fc29eb8790.jpg"): "CCW",
}
YOLO_WEIGHTS = {
    "yolo11n": ("yolo11n-seg.pt", "55ed65c56c91713d23e8402371c6c49a6fd84f257f7dce452e8d70e41dcbe152"),
    "yolov8n": ("yolov8n-seg.pt", "a7cd8f929e1903d78a12a48efecab430209f18dc46cb96c3599a5980c63c423c"),
}
YOLO_TRAIN = dict(
    epochs=60, patience=60, imgsz=640, batch=8, device=0, workers=2,
    amp=True, cache=False, seed=SEED, deterministic=True, val=True, plots=True,
    save=True, save_period=5, exist_ok=False, verbose=True, optimizer="AdamW",
    lr0=0.000128, momentum=0.9, weight_decay=0.0005, nbs=64, lrf=0.01,
    cos_lr=False, warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.0,
    mosaic=1.0, close_mosaic=10,
)
RCNN_TRAIN = dict(epochs=20, batch=2, workers=2, lr=0.005, momentum=0.9,
                  weight_decay=0.0005, milestones=[10, 15], gamma=0.1,
                  min_size=640, max_size=960, amp=True, save_period=5, max_minutes=105)
PROTOCOL = dict(dataset_split="validation", num_images=166, confidence_threshold=0.001,
                max_detections=100, mask_threshold=0.5, yolo_image_size=640,
                yolo_nms_iou=0.7, maskrcnn_min_size=640, maskrcnn_max_size=960,
                maskrcnn_nms_iou=0.5)
TRAIN_BASELINE = {"yolo11n": (60, 599.949), "yolov8n": (60, 571.242),
                  "maskrcnn": (20, 2238.738107909)}
PIPELINE_BASELINE = {"yolo11n":31.81647763899997,"yolov8n":19.78087676599995,"maskrcnn":40.2276660120001}
ROOT = Path("/kaggle/working/cpmk3_hard3") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ROOT.mkdir(parents=True, exist_ok=False)
DATA = ROOT / "dataset"
EXPERIMENTS = ROOT / "experiments"
EXPORTS = ROOT / "exports"
EXPORTS.mkdir()
MODEL_STATE = {key: {"training": "pending", "evaluation": "pending"} for key in RUN_TRAIN}
ACTIVE_RUNS = {}
ENVIRONMENT = dict(torch=torch.__version__, torchvision=torchvision.__version__,
                   ultralytics=ultralytics.__version__, gpu=torch.cuda.get_device_name(0),
                   cuda=torch.version.cuda, seed=SEED)
EXPERIMENT_SPEC = dict(yolo=YOLO_TRAIN, maskrcnn=RCNN_TRAIN, classes=CLASSES,
                       source_hashes=SOURCE_HASHES, protocol=PROTOCOL)

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
                    encoding="utf-8")
    temp.replace(path)

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8-sig"))

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def fingerprint(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(payload.encode()).hexdigest()

SPEC_HASH = fingerprint(EXPERIMENT_SPEC)
write_json(ROOT / "environment.json", ENVIRONMENT)
write_json(ROOT / "experiment_spec.json", EXPERIMENT_SPEC)

def save_status():
    write_json(ROOT / "status.json", MODEL_STATE)

def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def release_gpu():
    gc.collect()
    torch.cuda.empty_cache()

save_status()
print(ROOT, ENVIRONMENT)

## 3. Identifikasi data

Anotasi ketiga split diperiksa terhadap versi baseline. Sel persiapan membaca original test untuk memeriksa struktur dan mempertahankan split. Semua loader dan evaluasi di bawah hanya memakai train dan validation.

In [ ]:
def discover_dataset(override, input_root):
    if override is not None:
        candidates = [Path(override)]
    else:
        candidates = sorted({p.parent.parent for p in input_root.rglob("_annotations.coco.json")
                             if p.parent.name == "train" and sha256(p) == SOURCE_HASHES["train"]})
    if len(candidates) != 1:
        raise RuntimeError(f"Perlu satu dataset COCO versi baseline. Atur DATASET_ROOT. Kandidat: {candidates}")
    root = candidates[0]
    for split, digest in SOURCE_HASHES.items():
        path = root / split / "_annotations.coco.json"
        if not path.is_file() or sha256(path) != digest:
            raise ValueError(f"Anotasi {split} berbeda dari baseline: {path}")
    return root

COCO_ROOT = discover_dataset(DATASET_ROOT, INPUT_ROOT)
print("Dataset:", COCO_ROOT)

## 4. Koreksi anotasi dan persiapan Hard-3

Empat gambar telah memiliki orientasi piksel yang benar. Karena ukuran pada metadata COCO tertukar, yang diputar adalah titik poligonnya. Poligon target yang sama menjadi sumber label YOLO dan COCO. ID COCO asli 2, 4, dan 8 dipetakan ke ID Hard-3 COCO 1, 2, dan 3. Untuk YOLO, ID tersebut dikurangi satu.

In [ ]:
def rotate_polygon(points, width, height, direction):
    x, y = points.T
    if direction == "CW":
        return np.column_stack((height - y, x))
    if direction == "CCW":
        return np.column_stack((y, width - x))
    if direction is not None:
        raise ValueError(f"Rotasi tidak dikenal: {direction}")
    return points.copy()

def normalize_polygon(xy, size):
    result = xy / np.asarray(size, dtype=float)
    if not np.isfinite(result).all() or (result < 0).any() or (result > 1).any():
        raise ValueError("Poligon di luar batas gambar.")
    return result

def prepare_dataset(source_root, out):
    categories = [{"id": new, "name": name, "supercategory": "indoor"}
                  for new, name in CLASSES.values()]
    splits, repairs = {}, []
    for split, expected in EXPECTED.items():
        src_dir = source_root / split
        images_dir = out / split / "images"
        labels_dir = out / split / "labels"
        images_dir.mkdir(parents=True, exist_ok=False)
        labels_dir.mkdir()
        original = read_json(src_dir / "_annotations.coco.json")
        source_names = {row["id"]: row["name"] for row in original["categories"]}
        assert all(source_names[old] == name for old, (_, name) in CLASSES.items())
        metadata, sizes, labels, corrected = {}, {}, defaultdict(list), []
        seen_stems, seen_ids = set(), set()
        for image in original["images"]:
            name = image["file_name"]
            assert Path(name).name == name and image["id"] not in metadata, image
            assert Path(name).stem not in seen_stems, name
            seen_stems.add(Path(name).stem)
            src = src_dir / name
            with Image.open(src) as pic:
                actual = pic.size
                # YOLO dan Torchvision harus melihat orientasi piksel yang sama.
                assert pic.getexif().get(274, 1) == 1, f"Orientasi EXIF tidak netral: {name}"
                pic.verify()
            nominal = image["width"], image["height"]
            direction = ROTATIONS.get((split, name))
            if direction:
                assert actual == nominal[::-1] and actual != nominal, name
                repairs.append(dict(split=split, image_id=image["id"], file_name=name,
                                    rotation=direction, coco_size=nominal, image_size=actual))
            else:
                assert actual == nominal, (name, actual, nominal)
            metadata[image["id"]] = dict(image, width=actual[0], height=actual[1])
            sizes[image["id"]] = nominal, actual, direction
            shutil.copy2(src, images_dir / name)
        for ann in original["annotations"]:
            old = ann["category_id"]
            if old not in CLASSES:
                continue
            assert ann["id"] not in seen_ids and ann.get("iscrowd", 0) == 0, ann["id"]
            seen_ids.add(ann["id"])
            image_id = ann["image_id"]
            nominal, actual, direction = sizes[image_id]
            new, _ = CLASSES[old]
            polygon = ann["segmentation"]
            # Baseline menerima satu polygon per instance. Tolak format berbeda.
            assert isinstance(polygon, list) and len(polygon) == 1, ann["id"]
            coords = polygon[0]
            assert len(coords) >= 6 and len(coords) % 2 == 0, ann["id"]
            xy = rotate_polygon(np.asarray(coords, dtype=np.float64).reshape(-1, 2),
                                *nominal, direction)
            normalized = normalize_polygon(xy, actual)
            segment = [xy.ravel().tolist()]
            width, height = actual
            rle = mask_utils.merge(mask_utils.frPyObjects(segment, height, width))
            area = float(mask_utils.area(rle))
            x, y, w, h = map(float, mask_utils.toBbox(rle))
            assert area > 0 and min(x, y) >= 0 and min(w, h) > 0
            assert x + w <= width and y + h <= height, ann["id"]
            corrected.append(dict(ann, category_id=new, segmentation=segment,
                                  bbox=[x, y, w, h], area=area, iscrowd=0))
            coordinates = " ".join(f"{x:.10f}" for x in normalized.ravel())
            labels[image_id].append(f"{new - 1} {coordinates}")
        for image_id, image in metadata.items():
            lines = labels.get(image_id, [])
            path = labels_dir / (Path(image["file_name"]).stem + ".txt")
            path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
        positives = len(labels)
        assert (len(metadata), len(corrected), positives) == expected, (split, expected)
        assert len(list(images_dir.iterdir())) == len(list(labels_dir.iterdir())) == expected[0]
        gt = dict(info=original.get("info", {}), licenses=original.get("licenses", []),
                  images=list(metadata.values()), categories=categories, annotations=corrected)
        write_json(out / split / "hard3.coco.json", gt)
        splits[split] = dict(images=len(metadata), instances=len(corrected),
                             positive_images=positives, negative_images=len(metadata)-positives,
                             instances_by_class=dict(Counter(a["category_id"] for a in corrected)))
    assert {(row["split"], row["file_name"]) for row in repairs} == set(ROTATIONS)
    yaml_data = dict(path=str(out), train="train/images", val="valid/images",
                     test="test/images", nc=3, names=NAMES)
    (out / "data.yaml").write_text(yaml.safe_dump(yaml_data, sort_keys=False), encoding="utf-8")
    manifest = dict(source=str(source_root), source_sha256=SOURCE_HASHES, splits=splits,
                    orientation_repairs=repairs, test_used_for_training_or_evaluation=False,
                    class_mapping=[dict(original_coco=old, original_yolo=old-1, hard3_coco=new,
                                        hard3_yolo=new-1, name=name)
                                   for old, (new, name) in CLASSES.items()])
    write_json(ROOT / "dataset_manifest.json", manifest)
    return manifest

MANIFEST = prepare_dataset(COCO_ROOT, DATA)
DATA_YAML = DATA / "data.yaml"
VAL_ANN = DATA / "valid/hard3.coco.json"

## 5. Pemeriksaan data dan contoh anotasi

Train memiliki 60 gambar positif dan 337 negatif. Validation memiliki 19 gambar positif dan 147 negatif. Gambar yang berisi objek di luar Hard-3 tetap digunakan. Untuk Mask R-CNN, gambar tanpa target memakai tensor kosong dengan dimensi yang sesuai.

In [ ]:
class IndoorCOCODataset(Dataset):
    def __init__(self, split):
        if split not in {"train", "valid"}:
            raise ValueError("Loader notebook dibatasi pada train dan validation.")
        self.image_dir = DATA / split / "images"
        self.coco = COCO(str(DATA / split / "hard3.coco.json"))
        self.ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        meta = self.coco.imgs[image_id]
        with Image.open(self.image_dir / meta["file_name"]) as picture:
            image = to_tensor(picture.convert("RGB"))
        anns = self.coco.imgToAnns[image_id]
        boxes, masks = [], []
        for a in anns:
            x, y, w, h = a["bbox"]
            boxes.append([x, y, x + w, y + h])
            masks.append(self.coco.annToMask(a))
        n = len(anns)
        target = dict(
            boxes=torch.tensor(boxes, dtype=torch.float32).reshape(n, 4),
            labels=torch.tensor([a["category_id"] for a in anns], dtype=torch.int64),
            masks=torch.from_numpy(np.stack(masks)).to(torch.uint8) if n else
                  torch.zeros((0, meta["height"], meta["width"]), dtype=torch.uint8),
            image_id=torch.tensor(image_id, dtype=torch.int64),
            area=torch.tensor([a["area"] for a in anns], dtype=torch.float32),
            iscrowd=torch.zeros(n, dtype=torch.int64),
        )
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

VAL_DS = IndoorCOCODataset("valid")
example_ids = []
for cls in range(1, 4):
    example_ids.append(min(a["image_id"] for a in VAL_DS.coco.anns.values()
                           if a["category_id"] == cls))
example_ids.append(next(i for i in VAL_DS.ids if not VAL_DS.coco.imgToAnns[i]))
fig, axes = plt.subplots(2, 2, figsize=(9, 7), layout="constrained")
for axis, image_id in zip(axes.flat, example_ids):
    pixels, target = VAL_DS[VAL_DS.ids.index(image_id)]
    n = len(target["labels"])
    assert target["boxes"].shape == (n, 4)
    assert target["masks"].shape == (n, *pixels.shape[1:])
    axis.imshow(pixels.permute(1, 2, 0))
    for m in target["masks"]:
        axis.contour(m.numpy(), levels=[0.5], linewidths=1.3)
    names = ", ".join(NAMES[i-1] for i in target["labels"].tolist())
    axis.set_title(f"ID {image_id}: {names or 'tanpa instance target'}", fontsize=9)
    axis.axis("off")
fig.savefig(ROOT / "dataset_qa.png", dpi=150)
plt.show()
plt.close(fig)
display(pd.DataFrame.from_dict(MANIFEST["splits"], orient="index"))

## 6–7. Pelatihan kedua model YOLO

Training memakai satu GPU secara berurutan. Setiap percobaan memiliki folder sendiri. Hasil dianggap lengkap setelah log mencatat semua epoch yang direncanakan dan checkpoint wajib tersedia. ZIP individual menyimpan checkpoint, log, konfigurasi, dan QA data tanpa gambar dataset.

In [ ]:
import logging
from ultralytics.utils import LOGGER
from ultralytics.utils.downloads import attempt_download_asset

def checkpoint_names(key):
    ext = "pth" if key == "maskrcnn" else "pt"
    return (f"weights/best.{ext}", f"weights/last.{ext}")

def validate_training_artifacts(key, run):
    needed = ["results.csv", "summary.json", "config.json", *checkpoint_names(key)]
    needed += ["training_curves.png"] if key == "maskrcnn" else ["results.png", "args.yaml"]
    for name in needed:
        path = run / name
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Artefak wajib: {path}")
    summary = read_json(run / "summary.json")
    expected = RCNN_TRAIN["epochs"] if key == "maskrcnn" else YOLO_TRAIN["epochs"]
    if (summary["model"] != key or summary["epochs_completed"] != expected
            or summary["spec_sha256"] != SPEC_HASH
            or not math.isfinite(summary["training_seconds"])
            or summary["training_seconds"] <= 0):
        raise ValueError(f"Run tidak lengkap atau spesifikasinya berbeda: {run}")
    for name, digest in summary["checkpoint_sha256"].items():
        if sha256(run / name) != digest:
            raise ValueError(f"Checksum checkpoint berbeda: {run / name}")

def archive_verified(destination, files):
    if destination.exists():
        raise FileExistsError(destination)
    temporary = destination.with_suffix(".zip.tmp")
    expected = {}
    with ZipFile(temporary, "x", allowZip64=True) as z:
        for name, path in files:
            assert name not in expected and path.is_file(), (name, path)
            expected[name] = path.stat().st_size
            z.write(path, arcname=name, compress_type=ZIP_STORED if path.suffix in
                    {".pt", ".pth", ".png", ".jpg"} else ZIP_DEFLATED)
    with ZipFile(temporary) as z:
        assert {entry.filename: entry.file_size for entry in z.infolist()} == expected
        bad = z.testzip()
        if bad is not None:
            raise IOError(f"CRC ZIP gagal: {bad}")
    temporary.rename(destination)

def run_entries(run, prefix):
    return [(f"{prefix}/{p.relative_to(run).as_posix()}", p)
            for p in sorted(run.rglob("*")) if p.is_file()]

def common_entries():
    return [(f"provenance/{n}", ROOT / n) for n in
            ("environment.json", "experiment_spec.json", "dataset_manifest.json", "dataset_qa.png")]

def restore_model(key, source, run):
    prefix = f"model/{key}/"
    with ZipFile(source) as z:
        entries = [entry for entry in z.infolist() if entry.filename.startswith(prefix)
                   and not entry.is_dir()]
        if not entries:
            raise ValueError(f"ZIP tidak berisi {prefix}: {source}")
        for entry in entries:
            relative = PurePosixPath(entry.filename.removeprefix(prefix))
            if relative.is_absolute() or ".." in relative.parts or "\\" in str(relative):
                raise ValueError(entry.filename)
            path = run.joinpath(*relative.parts)
            path.parent.mkdir(parents=True, exist_ok=True)
            with z.open(entry) as incoming, path.open("xb") as outgoing:
                shutil.copyfileobj(incoming, outgoing)
    validate_training_artifacts(key, run)

def next_run(key):
    parent = EXPERIMENTS / key
    parent.mkdir(parents=True, exist_ok=True)
    attempt = 1
    while (parent / f"attempt_{attempt:02d}").exists():
        attempt += 1
    return parent / f"attempt_{attempt:02d}"

def finish_summary(key, run, summary):
    summary.update(model=key, spec_sha256=SPEC_HASH,
                   checkpoint_sha256={n: sha256(run / n) for n in checkpoint_names(key)})
    write_json(run / "summary.json", summary)

def run_training(key, trainer):
    state = MODEL_STATE[key]
    if state["training"] == "completed":
        print(f"{key}: training sudah lengkap.")
        return
    if not RUN_TRAIN[key] and key not in RECOVERY_ZIPS:
        state.update(training="skipped", evaluation="skipped")
        save_status()
        return
    run = next_run(key)
    state.update(training="running", evaluation="pending", run=str(run))
    save_status()
    try:
        seed_all()
        if key in RECOVERY_ZIPS:
            restore_model(key, Path(RECOVERY_ZIPS[key]), run)
            state["recovered_from"] = str(RECOVERY_ZIPS[key])
        else:
            trainer(key, run)
        validate_training_artifacts(key, run)
        destination = EXPORTS / f"{key}_{run.name}.zip"
        archive_verified(destination, common_entries() + run_entries(run, f"model/{key}"))
        ACTIVE_RUNS[key] = run
        state.update(training="completed", training_zip=str(destination))
    except Exception:
        message = traceback.format_exc()
        run.mkdir(parents=True, exist_ok=True)
        (run / "training_error.txt").write_text(message, encoding="utf-8")
        state.update(training="failed", evaluation="skipped", error=str(run / "training_error.txt"))
        print(message)
    finally:
        release_gpu()
        save_status()

def require_finite_yolo(trainer):
    loss = trainer.loss
    if not torch.isfinite(loss).all().item():
        raise FloatingPointError(f"Loss YOLO tidak finite pada epoch {trainer.epoch + 1}")

def train_yolo(key, run):
    name, expected_sha = YOLO_WEIGHTS[key]
    requested = Path(PRETRAINED_ROOT) / name if PRETRAINED_ROOT else ROOT / "pretrained" / name
    requested.parent.mkdir(parents=True, exist_ok=True)
    initial = Path(attempt_download_asset(str(requested))) if PRETRAINED_ROOT is None else requested
    if not initial.is_file() or sha256(initial) != expected_sha:
        raise ValueError(f"Bobot pralatih tidak cocok dengan baseline: {initial}")
    model = YOLO(str(initial), task="segment")
    model.add_callback("on_train_batch_end", require_finite_yolo)
    # Simpan log konsol di luar folder run sampai Ultralytics membuatnya.
    logfile = run.parent / f"{run.name}_console.log"
    handler = logging.FileHandler(logfile, encoding="utf-8")
    LOGGER.addHandler(handler)
    try:
        model.train(data=str(DATA_YAML), project=str(run.parent), name=run.name, **YOLO_TRAIN)
        actual_run = Path(model.trainer.save_dir).resolve()
        if actual_run != run.resolve():
            raise RuntimeError(f"Path output Ultralytics berbeda: {actual_run}")
    finally:
        LOGGER.removeHandler(handler)
        handler.close()
        run.mkdir(exist_ok=True)
        logfile.replace(run / "training.log")
        del model
    history = pd.read_csv(run / "results.csv")
    history.columns = history.columns.str.strip()
    required = ["epoch", "time", "metrics/mAP50-95(M)", "metrics/mAP50(M)",
                "metrics/mAP50-95(B)", "metrics/precision(M)", "metrics/recall(M)",
                "train/seg_loss", "val/seg_loss"]
    numeric = history[required].apply(pd.to_numeric, errors="raise")
    loss_cols = [c for c in history.columns if c.startswith(("train/", "val/")) and c.endswith("_loss")]
    if (numeric["epoch"].tolist() != list(range(1, YOLO_TRAIN["epochs"] + 1))
            or not np.isfinite(history[loss_cols].apply(pd.to_numeric).to_numpy()).all()
            or not np.isfinite(numeric.to_numpy()).all()):
        raise RuntimeError("Log epoch atau nilai metrik YOLO tidak valid.")
    peak = numeric.loc[numeric["metrics/mAP50-95(M)"].idxmax()]
    write_json(run / "config.json", dict(YOLO_TRAIN, pretrained=name, pretrained_sha256=expected_sha,
                                         classes=NAMES, evaluator="Ultralytics"))
    finish_summary(key, run, dict(epochs_completed=len(numeric),
                                  training_seconds=float(numeric["time"].iloc[-1]),
                                  training_time_scope="Ultralytics cumulative epoch log",
                                  highest_validation_mask_map_epoch=int(peak["epoch"]),
                                  highest_validation_mask_map=float(peak["metrics/mAP50-95(M)"]),
                                  checkpoint_selection="Ultralytics validation fitness"))

run_training("yolo11n", train_yolo)

In [ ]:
run_training("yolov8n", train_yolo)

## 8. Pelatihan Mask R-CNN

Seed, resize, SGD, scheduler, dan AMP mengikuti baseline Mask R-CNN. Selama training, pemilihan checkpoint memakai evaluator COCO yang sama seperti baseline Mask R-CNN, termasuk ambang skor bawaan Torchvision 0,05. Evaluasi bersama atas checkpoint terpilih memakai confidence 0,001 dan NMS 0,50 sesuai protokol unified COCOeval. Checkpoint berkala dibuat setiap lima epoch agar hasil antara dapat disimpan.

In [ ]:
def coco_evaluate(gt, records, kind):
    with redirect_stdout(io.StringIO()):
        if records:
            detected = gt.loadRes(deepcopy(records))
        else:
            detected = COCO()
            detected.dataset = dict(info={}, images=gt.dataset["images"],
                                    categories=gt.dataset["categories"], annotations=[])
            detected.createIndex()
        evaluator = COCOeval(gt, detected, kind)
        evaluator.params.imgIds = sorted(gt.getImgIds())
        evaluator.params.catIds = [1, 2, 3]
        evaluator.params.maxDets = [1, 10, PROTOCOL["max_detections"]]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return evaluator

def as_rle(mask):
    coded = mask_utils.encode(np.asfortranarray(mask, dtype=np.uint8))
    coded["counts"] = coded["counts"].decode("ascii")
    return coded

def append_detections(boxes, labels, scores, masks, image_id, shape, threshold, bbox, segm):
    if not (len(boxes) == len(labels) == len(scores) == len(masks)):
        raise ValueError("Panjang keluaran prediksi berbeda.")
    if len(boxes) > PROTOCOL["max_detections"]:
        raise ValueError("Batas deteksi per gambar terlampaui.")
    for (x1, y1, x2, y2), cls, score, mask in zip(boxes, labels, scores, masks):
        x1, y1, x2, y2, score = map(float, (x1, y1, x2, y2, score))
        if not np.isfinite([x1, y1, x2, y2, score]).all():
            raise FloatingPointError("Prediksi tidak finite.")
        if score < threshold:
            continue
        if int(cls) not in (1, 2, 3) or mask.shape != shape:
            raise ValueError(f"ID kelas atau ukuran mask tidak sesuai: {cls}, {mask.shape}, {shape}")
        h, w = shape
        x1, x2 = np.clip([x1, x2], 0, w)
        y1, y2 = np.clip([y1, y2], 0, h)
        if x2 <= x1 or y2 <= y1:
            continue
        common = dict(image_id=image_id, category_id=int(cls), score=score)
        box = [float(x1), float(y1), float(x2-x1), float(y2-y1)]
        bbox.append(dict(common, bbox=box))
        binary = mask >= PROTOCOL["mask_threshold"]
        if binary.any():
            segm.append(dict(common, segmentation=as_rle(binary)))

def coco_metrics(gt, box_predictions, mask_predictions):
    box = coco_evaluate(gt, box_predictions, "bbox")
    mask = coco_evaluate(gt, mask_predictions, "segm")
    metrics = dict(box_map50_95=float(box.stats[0]), box_map50=float(box.stats[1]),
                   mask_map50_95=float(mask.stats[0]), mask_map50=float(mask.stats[1]),
                   mask_map75=float(mask.stats[2]), mask_ar100=float(mask.stats[8]))
    per_class = []
    for idx, cls in enumerate(mask.params.catIds):
        data = mask.eval["precision"][:, :, idx, 0, -1]
        values = data[data >= 0]
        per_class.append(dict(hard3_coco_id=cls, class_name=NAMES[cls-1],
                              mask_ap50_95=float(values.mean()) if values.size else None))
    return metrics, per_class

def make_maskrcnn(pretrained, unified=False):
    args = dict(min_size=RCNN_TRAIN["min_size"], max_size=RCNN_TRAIN["max_size"],
                box_detections_per_img=100)
    if unified:
        args.update(box_score_thresh=PROTOCOL["confidence_threshold"],
                    box_nms_thresh=PROTOCOL["maskrcnn_nms_iou"])
    weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn(weights=weights, weights_backbone=None, **args)
    box_in = model.roi_heads.box_predictor.cls_score.in_features
    mask_in = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.box_predictor = FastRCNNPredictor(box_in, 4)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_in, 256, 4)
    assert model.roi_heads.box_predictor.cls_score.out_features == 4
    assert model.roi_heads.mask_predictor.mask_fcn_logits.out_channels == 4
    return model.to(DEVICE)

def validation_loader():
    return DataLoader(VAL_DS, batch_size=1, shuffle=False, num_workers=2,
                      pin_memory=True, collate_fn=collate_fn)

@torch.inference_mode()
def collect_maskrcnn(model, loader, threshold):
    box_predictions, mask_predictions, seen = [], [], set()
    model.eval()
    for images, targets in loader:
        outputs = model([im.to(DEVICE) for im in images])
        assert len(outputs) == len(images)
        for image, target, output in zip(images, targets, outputs):
            image_id = int(target["image_id"].item())
            assert image_id not in seen
            seen.add(image_id)
            masks = output["masks"].cpu().numpy()
            assert masks.shape == (len(output["labels"]), 1, *image.shape[-2:])
            append_detections(output["boxes"].cpu().numpy(), output["labels"].cpu().numpy(),
                              output["scores"].cpu().numpy(), masks[:, 0], image_id,
                              tuple(image.shape[-2:]), threshold, box_predictions, mask_predictions)
    assert seen == set(VAL_DS.ids)
    return box_predictions, mask_predictions

def save_rcnn_checkpoint(path, model, optimizer, scheduler, scaler, epoch, metrics):
    torch.save(dict(model_state_dict=model.state_dict(), optimizer_state_dict=optimizer.state_dict(),
                    scheduler_state_dict=scheduler.state_dict(), scaler_state_dict=scaler.state_dict(),
                    epoch=epoch, metrics=metrics, classes=NAMES), path)

def train_maskrcnn(key, run):
    weight_dir = run / "weights"
    weight_dir.mkdir(parents=True)
    cfg = RCNN_TRAIN
    write_json(run / "config.json", dict(cfg, pretrained="Torchvision COCO DEFAULT", classes=NAMES,
                                         foreground_classes=3, classes_with_background=4,
                                         train_validation_score_threshold=0.05))
    train_loader = DataLoader(IndoorCOCODataset("train"), batch_size=cfg["batch"], shuffle=True,
                              num_workers=cfg["workers"], pin_memory=True, collate_fn=collate_fn,
                              generator=torch.Generator().manual_seed(SEED))
    val_loader = validation_loader()
    model = make_maskrcnn(pretrained=True)
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                                lr=cfg["lr"], momentum=cfg["momentum"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, cfg["milestones"], gamma=cfg["gamma"])
    scaler = torch.amp.GradScaler("cuda", enabled=cfg["amp"])
    history, best = [], -math.inf
    start = time.perf_counter()
    with (run / "training.log").open("x", encoding="utf-8") as log:
        for epoch in range(1, cfg["epochs"] + 1):
            if epoch > 1 and time.perf_counter() - start >= cfg["max_minutes"] * 60:
                raise TimeoutError(f"Anggaran waktu baseline tercapai sebelum epoch {epoch}.")
            lr = optimizer.param_groups[0]["lr"]
            model.train()
            sums = Counter()
            train_start = time.perf_counter()
            for batch, (images, targets) in enumerate(train_loader, 1):
                images = [im.to(DEVICE) for im in images]
                targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=cfg["amp"]):
                    parts = model(images, targets)
                    loss = sum(parts.values())
                if not torch.isfinite(loss).item():
                    raise FloatingPointError(f"Loss Mask R-CNN nonfinite: epoch {epoch}, batch {batch}")
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                sums.update({k: float(v.detach().item()) for k, v in parts.items()})
            torch.cuda.synchronize()
            train_seconds = time.perf_counter() - train_start
            val_start = time.perf_counter()
            boxes, masks = collect_maskrcnn(model, val_loader, threshold=0.05)
            metrics, per_class = coco_metrics(VAL_DS.coco, boxes, masks)
            validation_seconds = time.perf_counter() - val_start
            scheduler.step()
            row = dict(epoch=epoch, time=time.perf_counter()-start,
                       train_seconds=train_seconds, validation_seconds=validation_seconds,
                       learning_rate=lr, train_total_loss=sum(sums.values())/len(train_loader),
                       **{k: v/len(train_loader) for k, v in sums.items()}, **metrics)
            history.append(row)
            pd.DataFrame(history).to_csv(run / "results.csv", index=False)
            if metrics["mask_map50_95"] > best:
                best, best_epoch = metrics["mask_map50_95"], epoch
                save_rcnn_checkpoint(weight_dir / "best.pth", model, optimizer, scheduler, scaler, epoch, metrics)
                pd.DataFrame(per_class).to_csv(run / "per_class_training_validation.csv", index=False)
            save_rcnn_checkpoint(weight_dir / "last.pth", model, optimizer, scheduler, scaler, epoch, metrics)
            if epoch % cfg["save_period"] == 0:
                shutil.copy2(weight_dir / "last.pth", weight_dir / f"epoch{epoch:03d}.pth")
            print(row, file=log, flush=True)
            print(f"Mask R-CNN epoch {epoch}/{cfg['epochs']} | loss {row['train_total_loss']:.4f} | mask AP {metrics['mask_map50_95']:.4f}")
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    axes[0].plot(h.epoch, h.train_total_loss)
    axes[0].set(xlabel="Epoch", ylabel="Loss train")
    axes[1].plot(h.epoch, 100 * h.mask_map50_95)
    axes[1].set(xlabel="Epoch", ylabel="Mask mAP50–95 validation (%)")
    fig.savefig(run / "training_curves.png", dpi=150)
    plt.close(fig)
    finish_summary(key, run, dict(epochs_completed=len(history), training_seconds=history[-1]["time"],
                                  training_time_scope="Mask R-CNN epoch loop including validation",
                                  best_epoch=best_epoch, highest_validation_mask_map=best,
                                  checkpoint_selection="validation mask mAP50–95"))

run_training("maskrcnn", train_maskrcnn)

## 9. Unified COCOeval pada validation

Ketiga checkpoint dinilai pada 166 gambar yang sama. Evaluator memakai confidence minimum 0,001, maksimum 100 deteksi per gambar, dan mask pada resolusi asli. YOLO memakai `imgsz=640`, NMS IoU 0,70, dan `retina_masks=True` untuk memulihkan ukuran mask asli. Mask R-CNN memakai resize 640–960, NMS IoU 0,50, dan mask threshold 0,50. Pada Ultralytics, threshold 0,50 dipakai untuk mask keluaran yang sudah biner. Durasi pipeline mencakup pembacaan gambar, prediksi, pemrosesan keluaran, dan encoding mask. Waktu pemuatan checkpoint, COCOeval, dan penulisan JSON metrik dicatat di luar ruang lingkup durasi pipeline.

In [ ]:
@torch.inference_mode()
def collect_yolo(model):
    metadata = {p["file_name"]: p for p in VAL_DS.coco.imgs.values()}
    inputs = [str(VAL_DS.image_dir / VAL_DS.coco.imgs[i]["file_name"]) for i in VAL_DS.ids]
    boxes, masks, seen = [], [], set()
    outputs = model.predict(source=inputs, stream=True, batch=1, device=0,
                            imgsz=PROTOCOL["yolo_image_size"], retina_masks=True,
                            conf=PROTOCOL["confidence_threshold"], iou=PROTOCOL["yolo_nms_iou"],
                            max_det=PROTOCOL["max_detections"], agnostic_nms=False,
                            rect=True, save=False, verbose=False)
    for result in outputs:
        meta = metadata[Path(result.path).name]
        image_id = meta["id"]
        assert image_id not in seen
        seen.add(image_id)
        shape = (meta["height"], meta["width"])
        assert result.orig_shape == shape, (meta["file_name"], result.orig_shape, shape)
        if result.boxes is None or not len(result.boxes):
            continue
        if result.masks is None or tuple(result.masks.data.shape[1:]) != shape:
            raise ValueError(f"Mask YOLO tidak pada resolusi asli: {meta['file_name']}")
        append_detections(result.boxes.xyxy.cpu().numpy(),
                          result.boxes.cls.cpu().numpy().astype(int)+1,
                          result.boxes.conf.cpu().numpy(), result.masks.data.cpu().numpy(),
                          image_id, shape, PROTOCOL["confidence_threshold"], boxes, masks)
    assert seen == set(VAL_DS.ids)
    return boxes, masks

def evaluate_checkpoint(key, run, out):
    checkpoint = run / checkpoint_names(key)[0]
    if key == "maskrcnn":
        model = make_maskrcnn(pretrained=False, unified=True)
        state = torch.load(checkpoint, map_location="cpu", weights_only=True)
        assert state["classes"] == NAMES and 1 <= state["epoch"] <= RCNN_TRAIN["epochs"]
        model.load_state_dict(state["model_state_dict"], strict=True)
        del state
        loader = validation_loader()
    else:
        model = YOLO(str(checkpoint), task="segment")
        assert model.task == "segment" and [model.names[i] for i in range(len(model.names))] == NAMES
        assert model.model.model[-1].nc == 3
    torch.cuda.synchronize()
    started = time.perf_counter()
    if key == "maskrcnn":
        boxes, masks = collect_maskrcnn(model, loader, PROTOCOL["confidence_threshold"])
    else:
        boxes, masks = collect_yolo(model)
    torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    del model
    write_json(out / "bbox.json", boxes)
    write_json(out / "segm.json", masks)
    metrics, per_class = coco_metrics(VAL_DS.coco, boxes, masks)
    assert all(v is not None and 0 <= v <= 1 for v in metrics.values())
    assert all(row["mask_ap50_95"] is not None for row in per_class)
    pd.DataFrame(per_class).to_csv(out / "per_class_ap.csv", index=False)
    write_json(out / "metrics.json", dict(metrics, model=key, inference_pipeline_seconds=seconds,
                                          checkpoint_sha256=sha256(checkpoint), **PROTOCOL))
    for name in ("bbox.json", "segm.json", "per_class_ap.csv", "metrics.json"):
        assert (out / name).stat().st_size > 0

def run_evaluation(key):
    state = MODEL_STATE[key]
    if state["evaluation"] == "completed":
        return
    if not RUN_EVALUATION or state["training"] != "completed":
        state["evaluation"] = "skipped"
        save_status()
        return
    run = ACTIVE_RUNS[key]
    idx = 1
    while (run / f"unified_validation_{idx:02d}").exists():
        idx += 1
    out = run / f"unified_validation_{idx:02d}"
    out.mkdir()
    state.update(evaluation="running", validation_dir=str(out))
    save_status()
    try:
        evaluate_checkpoint(key, run, out)
        state["evaluation"] = "completed"
    except Exception:
        message = traceback.format_exc()
        (out / "evaluation_error.txt").write_text(message, encoding="utf-8")
        state.update(evaluation="failed", evaluation_error=str(out / "evaluation_error.txt"))
        print(message)
    finally:
        release_gpu()
        save_status()

for key in RUN_TRAIN:
    run_evaluation(key)

## 10. Perbandingan all-class dan Hard-3

Tabel menggunakan tiga kelas yang sama. Kecocokan sumber baseline, parameter evaluasi yang tercatat, dan geometri ground truth tiga kelas diperiksa sebelum selisih dihitung. Baseline menyediakan AP dari model all-class, dengan pengaturan jumlah deteksi yang diterapkan saat model masih memprediksi 74 kelas. Susunan kelas, bobot head, dan persaingan deteksi merupakan bagian dari perbedaan eksperimen. Notebook evaluator all-class belum tersedia di repo, sehingga kesetaraan implementasi detail rasterisasi dan pascapemrosesan belum dapat dipastikan. Selisih pada validation bersifat deskriptif.

In [ ]:
BASELINE_FINGERPRINTS = {"config":"8d61941fbdfeb22a9b74e7b48dc154d73108c9025b86f4511e35adc9e5482feb","per_class":"6be50420eaf387c73534f101ceac59e3eac89ca486ac4b771d8f2e96e15d08c9"}
EXPECTED_VAL_SIGNATURE = "1328aacbf05d7e2666500494f7a17493f7859eaea326b12671fde23ba441d72b"

def validation_signature(gt, convert_ids):
    images = sorted((a["id"], a["file_name"], a["width"], a["height"])
                    for a in gt["images"])
    annotations = sorted(
        (a["id"], a["image_id"], convert_ids[a["category_id"]],
         [[round(float(x), 8) for x in polygon] for polygon in a["segmentation"]])
        for a in gt["annotations"] if a["category_id"] in convert_ids
    )
    return fingerprint([images, annotations])

def evaluate_baseline_payload(payload):
    cfg = json.loads(payload["evaluation_config.json"])
    rows = list(csv.DictReader(io.StringIO(payload["per_class_ap_comparison.csv"].decode("utf-8-sig"))))
    if dict(config=fingerprint(cfg), per_class=fingerprint(rows)) != BASELINE_FINGERPRINTS:
        raise ValueError("Artefak baseline berbeda dari hasil unified COCOeval yang diaudit.")
    assert all(cfg[k] == v for k, v in PROTOCOL.items())
    assert cfg["num_classes"] == 74 and cfg["retraining_performed"] is False
    assert cfg["test_set_evaluated"] is False
    signatures = {(r["image_id"], r["file_name"], r["rotation"]) for r in cfg["orientation_fixes"]}
    assert signatures == {(r["image_id"], r["file_name"], r["rotation"])
                          for r in MANIFEST["orientation_repairs"] if r["split"] == "valid"}
    assert validation_signature(read_json(VAL_ANN), {n: n for n in range(1, 4)}) == EXPECTED_VAL_SIGNATURE
    selected = {int(r["category_id"]): r for r in rows if int(r["category_id"]) in CLASSES}
    assert set(selected) == set(CLASSES)
    for old, (new, name) in CLASSES.items():
        r = selected[old]
        assert r["class_name"] == name
        assert int(r["train_instances"]) == MANIFEST["splits"]["train"]["instances_by_class"][new]
        assert int(r["val_instances"]) == MANIFEST["splits"]["valid"]["instances_by_class"][new]
        assert all(0 <= float(r[key+"_mask_ap"]) <= 1 for key in RUN_TRAIN)
    return selected

def baseline_payload(source):
    names = ("evaluation_config.json", "per_class_ap_comparison.csv")
    if source.is_dir():
        return {k: (source / k).read_bytes() for k in names}
    with ZipFile(source) as z:
        data = {}
        for basename in names:
            matches = [n for n in z.namelist() if PurePosixPath(n).name == basename]
            if len(matches) != 1:
                raise ValueError(f"Perlu satu {basename} dalam {source}")
            data[basename] = z.read(matches[0])
    return data

def discover_baseline():
    if BASELINE_SOURCE is not None:
        return Path(BASELINE_SOURCE)
    candidates = {p.parent for p in INPUT_ROOT.rglob("per_class_ap_comparison.csv")
                  if (p.parent / "evaluation_config.json").is_file()}
    candidates.update(p for p in INPUT_ROOT.rglob("*.zip")
                      if p.name.lower() in {"hard-3 selection.zip", "all_class_evaluation_outputs.zip"})
    if len(candidates) != 1:
        raise RuntimeError(f"Lampirkan satu sumber baseline dan isi BASELINE_SOURCE jika ambigu: {sorted(candidates)}")
    return next(iter(candidates))

COMPARISON = ROOT / "comparison"
COMPARISON.mkdir()
comparison_status = "unavailable"
try:
    source = discover_baseline()
    payload = baseline_payload(source)
    baseline = evaluate_baseline_payload(payload)
    for name, contents in payload.items():
        (COMPARISON / ("all_class_" + name)).write_bytes(contents)
    by_class, by_model = [], []
    for key in RUN_TRAIN:
        state = MODEL_STATE[key]
        if state["evaluation"] != "completed":
            continue
        out = Path(state["validation_dir"])
        ev = read_json(out / "metrics.json")
        train = read_json(ACTIVE_RUNS[key] / "summary.json")
        hard = pd.read_csv(out / "per_class_ap.csv").set_index("hard3_coco_id")
        model_rows = []
        for old, (new, name) in CLASSES.items():
            ref = float(baseline[old][key + "_mask_ap"])
            score = float(hard.loc[new, "mask_ap50_95"])
            assert hard.loc[new, "class_name"] == name
            model_rows.append(dict(model=key, class_name=name, original_coco_id=old,
                                   hard3_coco_id=new, all_class_ap_pct=100*ref,
                                   hard3_ap_pct=100*score, delta_pp=100*(score-ref)))
        by_class.extend(model_rows)
        ref_epochs, ref_time = TRAIN_BASELINE[key]
        by_model.append(dict(
            model=key, all_class_mean_ap_pct=np.mean([r["all_class_ap_pct"] for r in model_rows]),
            hard3_mean_ap_pct=np.mean([r["hard3_ap_pct"] for r in model_rows]),
            delta_pp=np.mean([r["delta_pp"] for r in model_rows]),
            all_class_epochs=ref_epochs, hard3_epochs=train["epochs_completed"],
            all_class_training_seconds=ref_time, hard3_training_seconds=train["training_seconds"],
            all_class_inference_pipeline_seconds=PIPELINE_BASELINE[key],
            hard3_inference_pipeline_seconds=ev["inference_pipeline_seconds"],
        ))
    if by_model:
        details = pd.DataFrame(by_class)
        means = pd.DataFrame(by_model)
        details.to_csv(COMPARISON / "per_class_comparison.csv", index=False)
        means.to_csv(COMPARISON / "model_comparison.csv", index=False)
        display(means.round(3))
        display(details.round(3))
        comparison_status = "complete" if len(by_model) == len(RUN_TRAIN) else "partial"
    write_json(COMPARISON / "protocol_check.json", dict(
        baseline_source=str(source), config_matches=True, foreground_ground_truth_matches=True,
        includes_test_metrics=False,
        limitation="Kesetaraan kode evaluator all-class tidak dapat diaudit tanpa notebook asli.",
        baseline_training_time_scope="Waktu kumulatif dari CSV baseline",
        baseline_training_source="experiments/<model>_all_classes/metrics/results.csv",
    ))
except Exception:
    error = traceback.format_exc()
    (COMPARISON / "comparison_error.txt").write_text(error, encoding="utf-8")
    print(error)

## 11. Grafik dan ekspor

ZIP gabungan menyimpan model yang lengkap, prediksi validation, tabel perbandingan jika tersedia, dan status setiap model. Jika model gagal atau dilewati, status gabungan menyatakan hasil parsial. ZIP tidak menyertakan file gambar dari dataset. Checkpoint dan ZIP yang sudah lengkap dapat dipasang kembali sebagai input Kaggle untuk run berikutnya.

In [ ]:
comparison_path = COMPARISON / "model_comparison.csv"
if comparison_path.is_file():
    matches = pd.read_csv(comparison_path).set_index("model")
    ax = matches[["all_class_mean_ap_pct", "hard3_mean_ap_pct"]].rename(columns={
        "all_class_mean_ap_pct": "All-class pada 3 kelas", "hard3_mean_ap_pct": "Hard-3",
    }).plot.bar(rot=0, color=["#356e9e", "#bd692f"], figsize=(8, 4))
    ax.set(ylabel="Mean Mask AP50–95 (%)", xlabel="Model", title="Perbandingan pada validation")
    ax.set_ylim(bottom=0)
    ax.figure.tight_layout()
    ax.figure.savefig(COMPARISON / "model_comparison.png", dpi=160)
    plt.show()
    plt.close(ax.figure)
    scores = pd.read_csv(COMPARISON / "per_class_comparison.csv")
    ax = scores.pivot(index="class_name", columns="model", values="delta_pp").plot.bar(rot=0)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(ylabel="Hard-3 − all-class (poin persentase)", xlabel="Kelas")
    ax.figure.tight_layout()
    ax.figure.savefig(COMPARISON / "per_class_delta.png", dpi=160)
    plt.show()
    plt.close(ax.figure)

all_complete = comparison_status == "complete" and all(
    s["training"] == "completed" and s["evaluation"] == "completed" for s in MODEL_STATE.values()
)
write_json(ROOT / "run_summary.json", dict(
    state="completed" if all_complete else "partial",
    models=MODEL_STATE, comparison=comparison_status, original_test_evaluated=False,
    personal_test_evaluated=False,
))
combined = common_entries() + [
    ("status.json", ROOT / "status.json"), ("run_summary.json", ROOT / "run_summary.json"),
    ("validation/hard3.coco.json", VAL_ANN),
] + run_entries(COMPARISON, "comparison")
for key, run in ACTIVE_RUNS.items():
    validate_training_artifacts(key, run)
    combined.extend(run_entries(run, f"model/{key}"))
for key, state in MODEL_STATE.items():
    if state["training"] == "failed" and state.get("error"):
        combined.append((f"errors/{key}.txt", Path(state["error"])))
final_zip = EXPORTS / "CPMK3_Hard3_All_Models.zip"
archive_verified(final_zip, combined)
print(f"Status keseluruhan: {'completed' if all_complete else 'partial'}")
print("ZIP gabungan:", final_zip)
display(pd.DataFrame.from_dict(MODEL_STATE, orient="index"))
if not all_complete:
    print("RUN PARSIAL: periksa status.json dan traceback. Simpan versi ini untuk memulihkan run yang lengkap.")